In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [18]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [19]:
len(documents)

72

In [20]:
from minsearch import Index

In [21]:
def build_index(documents):
    index = Index(
        text_fields=["content"],
        keyword_fields=["filename"]
    )
    index.fit(documents)
    return index

In [22]:
index=build_index(documents)

In [23]:
index.search("How does the agentic loop keep calling the model until it stops?",num_results=3)

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [30]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-07-24 14:50:08--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-07-24 14:50:09 (16.0 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [24]:
from rag_helper_pro import RAG
from openai import OpenAI
import os



In [25]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAG(
    index=index,
    llm_client=openai_client,
    model="openai/gpt-oss-20b"
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [26]:
answer

('**How the agentic loop keeps the model “thinking” until it’s done**\n\nIn the code shown in the lesson the agent’s behavior is wrapped in a classic *while‑true* loop:\n\n```python\nwhile True:\n    # 1. Ask the model (with the current message history)\n    response = openai_client.responses.create(...)\n\n    # 2. Append the model’s output to the conversation\n    messages.extend(response.output)\n\n    # 3. Walk through that output\n    has_function_calls = False\n    for item in response.output:\n        if item.type == "function_call":\n            # We got a tool call – run it and add its result\n            call_output = make_call(item)\n            messages.append(call_output)\n            has_function_calls = True\n        elif item.type == "message":\n            # Regular assistant reply – we can print it, store it, etc.\n            last_answer = item.content[0].text\n\n    # 4. Decide whether to keep looping\n    if has_function_calls == False:   # no more tool calls in th

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
len(chunks)

295

In [9]:
index=build_index(chunks)

In [14]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

assistant = RAG(
    index=index,
    llm_client=openai_client,
    model="openai/gpt-oss-20b"
)

answer = assistant.rag("How does the agentic loop keep calling the model until it stops?")

In [15]:
answer

('The agentic loop keeps iterating by repeatedly calling the model inside a `while True` block.  \nAfter each call it checks the model’s output for any *function_call* items.  \nIf at least one function call is found (`has_function_calls` is set to `True`), the loop executes those calls, appends their results to the message history, and then goes back to the top of the loop to ask the model again.  \n\nWhen the model returns a turn that contains **no** function calls (`has_function_calls` remains `False`), the exit condition `if has_function_calls == False: break` is triggered, and the loop terminates.  \n\nSo the loop continues until the model produces a final answer that doesn’t ask for any further tool usage. (Optional safety nets like a max‑iteration counter, token budget, or time limit can also be added, but the core stop condition is the absence of function calls.)',
 ResponseUsage(input_tokens=2382, input_tokens_details=InputTokensDetails(cache_write_tokens=None, cached_tokens=0

In [10]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [34]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=1,
        boost_dict={"filename": 1.0, "content": 0.9},
       
    )

In [35]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [36]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
instructions="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(client=openai_client,model="openai/gpt-oss-120b")
)

In [37]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG and explain?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


In [41]:
result

LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.", role='developer', phase=None, type=None), EasyInputMessage(content='How does the agentic loop work, and how is it different from plain RAG and explain?', role='user', phase=None, type=None), ResponseReasoningItem(id='resp_01kyad9b99e5ar0p0p34a13h95', summary=[], type='reasoning', content=[Content(text='We need to answer question about "agentic loop" vs plain RAG and "explain". Probably from a course material. Need to search the FAQ. We\'ll do multiple searches: "agentic loop", "agentic loop vs RAG", "agentic loop explain".', type='reasoning_text')], encrypted_content=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"agentic loop"}', call_id='fc_d5995c9b-db6e-467a-a3c2-82fc0c2d8951', name='search', type='function_call', id='fc_d5995c9b-db6e-467a-a3c2-82fc0c2d89